# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Praveen23-kk/FlyRank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Based on our baseline rules, we rank content items to review first. The top items are flagged with `refresh_and_protect` actions, driven by reason codes like `stale_visible_decay_risk`. This reason code translates to: "High visibility but stale" — meaning these pages currently receive traffic and rank well, but are old and at a high risk of decaying.

In [1]:
import pandas as pd

# Load the scored actions
df = pd.read_csv('../outputs/baseline_action_score.csv')

# Display the top 10 recommended actions with their readable reason codes
cols_to_show = ['content_id', 'client_id', 'action_label', 'reason_code', 'baseline_refresh_score']
print("Top Recommended Actions Queue:")
print(df[cols_to_show].head(10).to_string(index=False))

Top Recommended Actions Queue:
          content_id         client_id        action_label              reason_code  baseline_refresh_score
content_9532f197bbc8 client_4e07408562 refresh_and_protect stale_visible_decay_risk                0.941189
content_4d1fe5b32dc2 client_19581e27de refresh_and_protect stale_visible_decay_risk                0.934889
content_07f2e7a6f38a client_19581e27de refresh_and_protect stale_visible_decay_risk                0.934080
content_e5ae436f9a16 client_4e07408562 refresh_and_protect stale_visible_decay_risk                0.933606
content_3430a8b94511 client_19581e27de refresh_and_protect stale_visible_decay_risk                0.933559
content_cbd93118300b client_19581e27de refresh_and_protect stale_visible_decay_risk                0.933263
content_9c195417f6ef client_19581e27de refresh_and_protect stale_visible_decay_risk                0.932991
content_ba2acb4ebd04 client_19581e27de refresh_and_protect stale_visible_decay_risk                0.9316

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Use**: This playbook and the underlying scoring system are designed strictly for **decision support**. It ranks content to help human editors prioritize which pages look worth reviewing first.

**Limits**:
- The associations we observed in the historical cross-sectional data are **not causal**. We observed that stale, highly visible pages are associated with traffic declines, but this dataset does not prove that doing a content refresh will cause an increase or recovery in traffic.
- Never use these rankings to automate content generation or blind updates.

In [2]:
# No code is strictly necessary here. Leaving empty or printing a small summary statement.
print("Limits: Decision support only. Not causal.")

Limits: Decision support only. Not causal.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated based on these scores.*

**Human Review Requirements**:
- **Verify actual staleness**: A person must check if the content is factually out-of-date, broken, or misaligned, rather than just old by timestamp.
- **Intent Match**: Check if the content still matches the user intent before rewriting. 

**The No-Go List**:
- **Do not automate rewrites**: Never automatically rewrite a page based solely on its position in this queue.
- **Do not blindly update timestamps**: If the content is functionally complete and accurate, changing the date just to "freshen" it is a no-go.

In [3]:
# No code necessary.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The playbook relies on the current distribution of metrics. You must trigger a review or retraining of the baseline rules if:
- The underlying dataset becomes older than 90 days.
- There is a noticeable drift in the baseline CTR or the overall baseline decline rate.
- Human reviewers consistently report that flagged `stale_visible_decay_risk` items are actually fully accurate and up to date.

In [4]:
import json

# Load baseline metrics to note the current baseline decline rate for monitoring
with open('../outputs/baseline_metrics.json', 'r') as f:
    metrics = json.load(f)

print(f"Monitor baseline drift against the current base decline rate: {metrics['base_decline_rate']:.2%}")

Monitor baseline drift against the current base decline rate: 54.21%


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ so they stay out of git via the CI leak guard.*

We export the top 100 recommended actions to `work/outputs/action_queue.csv`. This file remains safely out of git via the CI leak guard. These exported files and figures will serve as the foundation for the final research paper.

In [5]:
import os

# Ensure directories exist
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

# Export the top 100 queue
top_100_queue = df.head(100)
top_100_queue.to_csv('../outputs/action_queue.csv', index=False)
print("Saved top 100 action queue to ../outputs/action_queue.csv")

# Note: baseline_metrics.json already exists in outputs, we assume no new metrics are generated here.

Saved top 100 action queue to ../outputs/action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled out
- [x] The intended use is clearly marked as decision support, not causal automation
- [x] The no-go list explicitly bans blind automation
- [x] The code runs clean, top to bottom
- [x] Data boundaries (90-day expiry) are documented in the monitoring section
- [x] Exports are saved safely to `work/outputs/`
